# Experiment 05 — Exogenous features: paydays & local geography

Tests two new **exogenous** feature groups (created in notebook 02, candidates only — the main pipeline keeps using its old explicit feature list):

- `payday`: `day_of_month`, `is_payday` (the 15th and the last day of the month — wage days in Ecuador), `days_since_payday`, `days_to_payday`;
- `geo`: `city`, `state` of the store + `is_holiday_local`, `is_holiday_regional` (holidays matched to the store's location).

Both groups are known exactly for the test period (pure calendar/geography), so they add no leakage and no error accumulation in the iterative loop — the forecasting engine of experiment 04 is reused unchanged.

**Design.** Clean A/B on the same 3 honest validation windows as experiment 04: the only difference between variants is the feature list. Feature groups are toggled via a registry (`FEATURE_GROUPS` / `VARIANTS`); model caches are keyed by a hash of the active feature set, so toggling groups can never silently load a stale model. Baselines (`base` = the old feature set, and `base + 25% per-family blend`) are **reused from experiment 04's cache** — identical engine, data, params and seed. The gate is unchanged: a variant must beat its baseline on **all 3 windows**.

> 🇷🇺 Эксперимент с двумя новыми экзогенными группами фич: `payday` (зарплатные дни — 15-е и конец месяца) и `geo` (город/штат магазина + локальные/региональные праздники). Обе группы точно известны для теста — без утечек и без накопления ошибки. Чистое A/B на тех же 3 честных окнах, что в эксперименте 04: варианты отличаются только списком фич; группы включаются через реестр, кэш моделей привязан к хэшу набора фич. Гейт прежний: вариант обязан побить свой бейзлайн на всех 3 окнах.

In [1]:
import time

import os
import hashlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

In [2]:
_t0 = time.time()

SMOKE = os.environ.get("EXP05_SMOKE") == "1"
N_EST = 60 if SMOKE else 4155
SFX = "_smoke" if SMOKE else ""

MODELS_DIR = f"../models/exp05{SFX}"
ART_DIR    = f"../artifacts/exp05{SFX}"
EXP04_MODELS = f"../models/exp04{SFX}"     # per-family models reused from experiment 04
EXP04_ART    = f"../artifacts/exp04{SFX}"  # cached baseline predictions from experiment 04
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

LGBM_PARAMS = dict(
    n_estimators=N_EST,
    learning_rate=0.01,
    num_leaves=255,
    min_child_samples=30,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# Feature registry: groups are toggled, never hand-edited lists
# Реестр фич: включаются/выключаются группы, а не редактируются списки руками
FEATURE_GROUPS = {
    "calendar":     ["day_of_week", "month", "year", "is_weekend",
                     "day_of_year", "week_of_year", "date_index"],
    "fourier":      ["sin_day", "cos_day", "sin_week", "cos_week"],
    "lags":         ["lag_1", "lag_2", "lag_3", "lag_4", "lag_5", "lag_6",
                     "lag_7", "lag_14", "lag_21", "lag_28", "lag_42", "lag_56",
                     "lag_364", "lag_365"],
    "rollings":     ["rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_364"],
    "oil":          ["dcoilwtico", "dcoilwtico_ma7", "dcoilwtico_ma28"],
    "promo":        ["onpromotion", "onpromotion_ma7", "onpromotion_ma28"],
    "transactions": [f"transactions_lag_{l}" for l in range(16, 24)],
    "events":       ["is_holiday_national", "day_before_holiday",
                     "is_black_friday", "is_cyber_monday", "is_terremoto",
                     "is_navidad", "is_dia_madre", "is_futbol",
                     "is_dia_trabajo", "is_primer_dia", "is_dia_difuntos", "work_day"],
    "store":        ["store_nbr", "store_type", "cluster", "family"],
    # new candidate groups / новые группы-кандидаты
    "payday":       ["day_of_month", "is_payday", "days_since_payday", "days_to_payday"],
    "geo":          ["city", "state", "is_holiday_local", "is_holiday_regional"],
}

BASE_GROUPS = ["calendar", "fourier", "lags", "rollings", "oil", "promo",
               "transactions", "events", "store"]  # == FEATURES_V2 of notebooks 03/04

VARIANTS = {
    "base":       BASE_GROUPS,
    "payday":     BASE_GROUPS + ["payday"],
    "geo":        BASE_GROUPS + ["geo"],
    "payday_geo": BASE_GROUPS + ["payday", "geo"],
}

ALL_CAT_CANDIDATES = ["store_nbr", "store_type", "cluster", "family",
                      "day_of_week", "month", "city", "state"]


def feats_of(groups):
    return [f for g in groups for f in FEATURE_GROUPS[g]]


def cats_of(feats):
    return [c for c in ALL_CAT_CANDIDATES if c in feats]


def tag_of(groups):
    """Cache key: models are tied to the exact feature set / Ключ кэша: модель привязана к точному набору фич."""
    return hashlib.md5(",".join(feats_of(groups)).encode()).hexdigest()[:8]


ALL_FEATURES = feats_of(list(FEATURE_GROUPS))   # superset for building X once per day
BASE_FEATURES = feats_of(BASE_GROUPS)

WINDOWS = [
    ("W1", pd.Timestamp("2017-06-15"), pd.Timestamp("2017-06-30")),
    ("W2", pd.Timestamp("2017-07-15"), pd.Timestamp("2017-07-30")),
    ("W3", pd.Timestamp("2017-07-31"), pd.Timestamp("2017-08-15")),
]
WNAMES = [w for w, _, _ in WINDOWS]
TEST_START, TEST_END = pd.Timestamp("2017-08-16"), pd.Timestamp("2017-08-31")

for v, groups in VARIANTS.items():
    print(f"{v:11s} {len(feats_of(groups))} features, tag={tag_of(groups)}")
print(f"SMOKE={SMOKE}, n_estimators={N_EST}")

base        59 features, tag=c63a4fd0
payday      63 features, tag=2f084c92
geo         63 features, tag=b2842606
payday_geo  67 features, tag=1b6260a5
SMOKE=False, n_estimators=4155


In [3]:
# Load the feature superset and reshape sales into a (dates x series) matrix (same as experiment 04)
# Загружаем супермножество фич и укладываем продажи в матрицу (даты x ряды), как в эксперименте 04
DF = (
    pd.read_parquet("../artifacts/features.parquet")
    .sort_values(["date", "store_nbr", "family"])
    .reset_index(drop=True)
)
missing = [c for c in ALL_FEATURES if c not in DF.columns]
assert not missing, f"run notebook 02 first, missing columns: {missing}"

DATES = np.sort(DF["date"].unique())
N_DATES = len(DATES)
N_SERIES = DF.groupby("date").size().iloc[0]
assert len(DF) == N_DATES * N_SERIES, "calendar is not rectangular"
DATE_IDX = {d: i for i, d in enumerate(DATES)}

ref = DF.iloc[:N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
chk = DF.iloc[1000 * N_SERIES:1001 * N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
assert ref.equals(chk), "series order differs between dates"

FAMILIES = sorted(DF["family"].astype(str).unique())
FAM_ARR = ref["family"].astype(str).values
SALES_TRUE = DF["sales"].to_numpy(dtype=float).reshape(N_DATES, N_SERIES)

print(f"{N_DATES} dates x {N_SERIES} series; superset of {len(ALL_FEATURES)} features")

1704 dates x 1782 series; superset of 67 features


In [4]:
# Engine (experiment 04 engine, parameterized by feature set)
# X is built once per day over the feature superset; every predictor slices its own columns.
# Движок (движок эксперимента 04, параметризованный набором фич)
# X строится раз в день по супермножеству фич; каждый предиктор берёт свои колонки.
LAGS  = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]
ROLLS = [7, 14, 28]


def rmsle_mat(P, Y):
    return float(np.sqrt(np.mean((np.log1p(np.clip(P, 0, None)) - np.log1p(Y)) ** 2)))


def dynamic_features(sales_mat, i):
    feat = {}
    for k in LAGS:
        feat[f"lag_{k}"] = sales_mat[i - k]
    for w in ROLLS:
        feat[f"rolling_mean_{w}"] = np.nanmean(sales_mat[i - w:i], axis=0)
    win = sales_mat[i - 364:i]
    cnt = (~np.isnan(win)).sum(axis=0)
    rm = np.full(N_SERIES, np.nan)
    ok = cnt >= 30
    if ok.any():
        rm[ok] = np.nanmean(win[:, ok], axis=0)
    feat["rolling_mean_364"] = rm
    return feat


def x_for_day(i, sales_mat):
    X = DF.iloc[i * N_SERIES:(i + 1) * N_SERIES][ALL_FEATURES].copy()
    for c, v in dynamic_features(sales_mat, i).items():
        X[c] = v
    for col in ALL_CAT_CANDIDATES:
        X[col] = X[col].astype("category")
    return X


def zero_rule_mask(first_idx):
    return np.nansum(SALES_TRUE[first_idx - 21:first_idx], axis=0) == 0


def run_iterative(predict_day, window_dates, zmask):
    sm = SALES_TRUE.copy()
    idxs = [DATE_IDX[d] for d in window_dates]
    sm[idxs, :] = np.nan
    P = np.empty((len(idxs), N_SERIES))
    for j, i in enumerate(idxs):
        p = np.clip(predict_day(x_for_day(i, sm)), 0, None)
        p[zmask] = 0.0
        sm[i] = p
        P[j] = p
    return P


def fit_lgbm(sub, feats, cats):
    X = sub[feats].copy()
    y = np.log1p(sub["sales"])
    for col in cats:
        X[col] = X[col].astype("category")
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X, y)
    return m


def train_global(train_from, cutoff, groups, tag):
    """Cache name carries the feature-set hash — toggling groups can never load a stale model.
    Имя кэша содержит хэш набора фич — переключение групп не подгрузит устаревшую модель."""
    feats, cats = feats_of(groups), cats_of(feats_of(groups))
    path = os.path.join(MODELS_DIR, f"global_{tag}_{tag_of(groups)}.joblib")
    if os.path.exists(path):
        return joblib.load(path), feats
    sub = DF[(DF["date"] >= train_from) & (DF["date"] < cutoff) & DF["sales"].notna()]
    t = time.time()
    m = fit_lgbm(sub, feats, cats)
    joblib.dump(m, path)
    print(f"  trained global_{tag}_{tag_of(groups)}: {len(sub):,} rows, {time.time() - t:.0f}s")
    return m, feats


def load_per_family(tag):
    """Per-family models trained in experiment 04 (base feature set) — reused, not retrained.
    Per-family модели из эксперимента 04 (базовый набор фич) — переиспользуются, не переобучаются."""
    models = {}
    for fam in FAMILIES:
        slug = fam.replace("/", "_").replace(" ", "_")
        path = os.path.join(EXP04_MODELS, f"perfam_{tag}_{slug}.joblib")
        assert os.path.exists(path), f"run experiment 04 first: {path}"
        models[fam] = joblib.load(path)
    return models


def predictor_global(model, feats):
    def f(X):
        return np.expm1(model.predict(X[feats]))
    return f


def predictor_per_family(models, feats):
    def f(X):
        fam = X["family"].astype(str).values
        out = np.empty(len(X))
        Xs = X[feats]
        for name, m in models.items():
            mask = fam == name
            out[mask] = np.expm1(m.predict(Xs[mask]))
        return out
    return f


def predictor_blend(f1, f2, w1):
    def f(X):
        return w1 * f1(X) + (1.0 - w1) * f2(X)
    return f

In [5]:
# Sanity checks: matrix features match notebook 02; new features look right on known dates
# Проверки: матричные фичи совпадают с 02; новые фичи корректны на известных датах
i = DATE_IDX[pd.Timestamp("2017-05-10")]
feat = dynamic_features(SALES_TRUE, i)
block = DF.iloc[i * N_SERIES:(i + 1) * N_SERIES]
for c in ["lag_1", "lag_7", "lag_364", "rolling_mean_7", "rolling_mean_364"]:
    assert np.allclose(np.asarray(feat[c], float), block[c].to_numpy(float),
                       equal_nan=True, atol=1e-6), f"mismatch in {c}"

chk = DF[DF["date"] == pd.Timestamp("2017-08-15")].iloc[0]
assert chk["is_payday"] == 1 and chk["days_since_payday"] == 0 and chk["day_of_month"] == 15
chk = DF[DF["date"] == pd.Timestamp("2017-08-31")].iloc[0]
assert chk["is_payday"] == 1 and chk["days_to_payday"] == 0
chk = DF[DF["date"] == pd.Timestamp("2017-08-20")].iloc[0]
assert chk["is_payday"] == 0 and chk["days_since_payday"] == 5 and chk["days_to_payday"] == 11
assert DF["is_holiday_local"].sum() > 0 and DF["is_holiday_regional"].sum() > 0
print("OK: engine matches notebook 02; payday/geo features verified")

OK: engine matches notebook 02; payday/geo features verified


In [6]:
# Main runs
# base / base_blend25 are taken from experiment 04's cache (identical engine and seed);
# for each new variant: train an honest-cutoff global model + a 25% per-family blend (per-family
# models reused from experiment 04 — they use the base feature set, which is fine inside a blend).
# Основные прогоны
# base / base_blend25 берутся из кэша эксперимента 04 (движок и seed идентичны);
# для каждого нового варианта: глобальная модель с честной отсечкой + бленд 25% per-family
# (per-family модели переиспользуются из 04 — внутри бленда их базовый набор фич допустим).
RESULTS = {}
rows = []

for wname, ws, we in WINDOWS:
    print(f"\n=== {wname}: {ws.date()} .. {we.date()} ===")
    wdates = [d for d in DATES if ws <= d <= we]
    assert len(wdates) == 16
    si = DATE_IDX[wdates[0]]
    zmask = zero_rule_mask(si)
    Y = SALES_TRUE[[DATE_IDX[d] for d in wdates]]

    # Baselines from experiment 04 / Бейзлайны из эксперимента 04
    for scheme, src in [("base", "global_2016"), ("base_blend25", "blend_pf25_g16")]:
        P = np.load(os.path.join(EXP04_ART, f"pred_{wname}__{src}.npy"))
        RESULTS[(wname, scheme)] = P
        sc = rmsle_mat(P, Y)
        rows.append({"window": wname, "scheme": scheme, "rmsle": sc})
        print(f"  {scheme:22s} RMSLE={sc:.5f} (exp04 cache)")

    pf = load_per_family(wname)
    fpf = predictor_per_family(pf, BASE_FEATURES)

    for vname in ["payday", "geo", "payday_geo"]:
        model, feats = train_global(pd.Timestamp("2016-01-01"), ws, VARIANTS[vname], f"2016_{wname}")
        fg = predictor_global(model, feats)
        for scheme, fn in [(vname, fg),
                           (f"{vname}_blend25", predictor_blend(fpf, fg, 0.25))]:
            cache = os.path.join(ART_DIR, f"pred_{wname}__{scheme}.npy")
            t = time.time()
            if os.path.exists(cache):
                P = np.load(cache)
            else:
                P = run_iterative(fn, wdates, zmask)
                np.save(cache, P)
            RESULTS[(wname, scheme)] = P
            sc = rmsle_mat(P, Y)
            rows.append({"window": wname, "scheme": scheme, "rmsle": sc})
            print(f"  {scheme:22s} RMSLE={sc:.5f} ({time.time() - t:.0f}s)")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(ART_DIR, "results.csv"), index=False)
print(f"\nElapsed: {(time.time() - _t0) / 60:.1f} min")


=== W1: 2017-06-15 .. 2017-06-30 ===
  base                   RMSLE=0.38302 (exp04 cache)
  base_blend25           RMSLE=0.38195 (exp04 cache)
  payday                 RMSLE=0.38142 (0s)
  payday_blend25         RMSLE=0.38086 (0s)
  geo                    RMSLE=0.38272 (0s)
  geo_blend25            RMSLE=0.38173 (0s)
  payday_geo             RMSLE=0.38216 (0s)
  payday_geo_blend25     RMSLE=0.38120 (0s)

=== W2: 2017-07-15 .. 2017-07-30 ===
  base                   RMSLE=0.38801 (exp04 cache)
  base_blend25           RMSLE=0.38681 (exp04 cache)
  payday                 RMSLE=0.38860 (0s)
  payday_blend25         RMSLE=0.38701 (0s)
  geo                    RMSLE=0.38783 (0s)
  geo_blend25            RMSLE=0.38652 (0s)
  payday_geo             RMSLE=0.38867 (0s)
  payday_geo_blend25     RMSLE=0.38703 (0s)

=== W3: 2017-07-31 .. 2017-08-15 ===
  base                   RMSLE=0.40490 (exp04 cache)
  base_blend25           RMSLE=0.40261 (exp04 cache)
  payday                 RMSLE=0.40623 (

In [7]:
# Summary and gate
# A pure variant competes against `base`, a blend variant against `base_blend25`;
# the candidate must win on ALL 3 windows. Among gated candidates the best mean wins.
# Сводка и гейт
# Чистый вариант сравнивается с `base`, бленд — с `base_blend25`;
# кандидат обязан выиграть на ВСЕХ 3 окнах. Среди прошедших гейт побеждает лучшее среднее.
piv = res.pivot(index="scheme", columns="window", values="rmsle")
piv["mean"] = piv[WNAMES].mean(axis=1)
piv = piv.sort_values("mean")
display(piv.round(5))

def gated(cand, baseline):
    return all(piv.loc[cand, w] < piv.loc[baseline, w] for w in WNAMES)

candidates = {}
for v in ["payday", "geo", "payday_geo"]:
    candidates[v] = gated(v, "base")
    candidates[f"{v}_blend25"] = gated(f"{v}_blend25", "base_blend25")

passed = [c for c, ok in candidates.items() if ok]
print("Gate passed by:", passed if passed else "NOBODY")

FINAL_PURE  = min((c for c in passed if not c.endswith("_blend25")), key=lambda c: piv.loc[c, "mean"], default=None)
FINAL_BLEND = min((c for c in passed if c.endswith("_blend25")),     key=lambda c: piv.loc[c, "mean"], default=None)
print(f"best gated pure:  {FINAL_PURE}")
print(f"best gated blend: {FINAL_BLEND}")
for c, base in [(FINAL_PURE, "base"), (FINAL_BLEND, "base_blend25")]:
    if c:
        print(f"  {c}: mean {piv.loc[c, 'mean']:.5f} vs {base} {piv.loc[base, 'mean']:.5f} "
              f"(delta {piv.loc[base, 'mean'] - piv.loc[c, 'mean']:+.5f})")

window,W1,W2,W3,mean
scheme,,,,
payday_geo_blend25,0.38120,0.38703,0.40118,0.38980
payday_blend25,0.38086,0.38701,0.40187,0.38991
geo_blend25,0.38173,0.38652,0.40187,0.39004
base_blend25,0.38195,0.38681,0.40261,0.39046
geo,0.38272,0.38783,0.40369,0.39141
base,0.38302,0.38801,0.40490,0.39198
payday_geo,0.38216,0.38867,0.40543,0.39208
payday,0.38142,0.38860,0.40623,0.39208


Gate passed by: ['geo', 'geo_blend25']
best gated pure:  geo
best gated blend: geo_blend25
  geo: mean 0.39141 vs base 0.39198 (delta +0.00056)
  geo_blend25: mean 0.39004 vs base_blend25 0.39046 (delta +0.00042)


In [8]:
# Final: retrain gated winners on the full train and build submission files
# Финал: переобучение победителей на всём train и сборка файлов сабмишна
test_dates = [d for d in DATES if TEST_START <= d <= TEST_END]
assert len(test_dates) == 16
zmask_test = zero_rule_mask(DATE_IDX[test_dates[0]])
test = pd.read_csv("../data/test.csv", parse_dates=["date"])


def make_submission(P_test, name):
    long = pd.concat(
        [pd.DataFrame({"date": d, "store_nbr": ref["store_nbr"].values,
                       "family": ref["family"].values, "sales": P_test[j]})
         for j, d in enumerate(test_dates)],
        ignore_index=True,
    )
    sub = test.merge(long, on=["date", "store_nbr", "family"], how="left")[["id", "sales"]]
    assert sub["sales"].notna().all()
    path = os.path.join(ART_DIR, f"submission_{name}.csv") if SMOKE else f"../submission_05_{name}.csv"
    sub.to_csv(path, index=False)
    print(f"Saved: {path}")


finals = [c for c in [FINAL_PURE, FINAL_BLEND] if c]
if not finals:
    print("No candidate passed the gate — no submission files / Никто не прошёл гейт — файлы не создаются")

pf_full = load_per_family("FULL") if any(c.endswith("_blend25") for c in finals) else None

for cand in finals:
    vname = cand.replace("_blend25", "")
    model, feats = train_global(pd.Timestamp("2016-01-01"), TEST_START, VARIANTS[vname], "2016_FULL")
    fg = predictor_global(model, feats)
    fn = predictor_blend(predictor_per_family(pf_full, BASE_FEATURES), fg, 0.25) if cand.endswith("_blend25") else fg
    cache = os.path.join(ART_DIR, f"pred_TEST__{cand}.npy")
    if os.path.exists(cache):
        P_test = np.load(cache)
    else:
        P_test = run_iterative(fn, test_dates, zmask_test)
        np.save(cache, P_test)
    print(f"{cand}: test total = {P_test.sum():,.0f}")
    make_submission(P_test, cand)

print(f"Total runtime: {(time.time() - _t0) / 60:.1f} min")

geo: test total = 12,001,152
Saved: ../submission_05_geo.csv
geo_blend25: test total = 12,060,875
Saved: ../submission_05_geo_blend25.csv
Total runtime: 1.1 min


## Conclusions

**`geo` passes the gate, `payday` does not.**

- **`geo` (city/state + local & regional holidays): accepted.** It beats its baseline on all 3 windows, both on its own (mean 0.39141 vs 0.39198) and inside the 25% per-family blend (0.39004 vs 0.39046). The gain is small (~0.0005) but consistent across windows — exactly what the gate checks for. Local holidays matched to the store's city carry signal that a single national flag cannot.
- **`payday` (day-of-month / wage-day features): rejected.** This is an honest negative result. It gives the best single-window score of the whole experiment (W1: 0.38086 in the blend) but loses on W2 and W3. A one-window check would have accepted it; three windows show it is mostly noise, and it may overlap with seasonality features that already exist (`day_of_year` and the Fourier terms already cover much of the monthly cycle).
- `payday_geo_blend25` has the best mean (0.38980) but fails the gate on W2 — the mean is pulled up by payday's strong W1 alone. The gate is designed to reject exactly this kind of candidate.

**Files produced:** `submission_05_geo.csv` (global + geo) and `submission_05_geo_blend25.csv` (25% per-family + 75% global+geo — the best scheme that passed the gate).

**Note on the leaderboard.** Experiment 04 showed that gains tied to *which years the model trains on* do not transfer to the public score. The gains here come from *features* (same training data, same cutoffs), so they should transfer better — but at +0.0005 the effect may still be within public-test noise.

## Выводы

**`geo` проходит гейт, `payday` — нет.**

- **`geo` (город/штат + локальные и региональные праздники): принят.** Лучше своего бейзлайна на всех 3 окнах — и сам по себе (среднее 0.39141 против 0.39198), и внутри бленда 25% per-family (0.39004 против 0.39046). Выигрыш мал (~0.0005), но устойчив по окнам — именно это проверяет гейт. Локальные праздники, привязанные к городу магазина, несут сигнал, которого один национальный флаг дать не может.
- **`payday` (день месяца / зарплатные дни): отклонён.** Это честный отрицательный результат. Он даёт лучший результат на одном окне за весь эксперимент (W1: 0.38086 в бленде), но проигрывает на W2 и W3. Проверка по одному окну приняла бы его; три окна показывают, что это в основном шум, и он может пересекаться с уже имеющимися сезонными признаками (`day_of_year` и Fourier уже покрывают большую часть месячного цикла).
- `payday_geo_blend25` имеет лучшее среднее (0.38980), но проваливает гейт на W2 — среднее вытягивает только сильный W1 у payday. Гейт как раз и создан, чтобы отклонять такие кандидаты.

**Созданные файлы:** `submission_05_geo.csv` (global + geo) и `submission_05_geo_blend25.csv` (25% per-family + 75% global+geo — лучшая схема, прошедшая гейт).

**Замечание про лидерборд.** Эксперимент 04 показал, что выигрыши, связанные с тем, *на каких годах обучается модель*, не переносятся на публичный скор. Здесь выигрыш даёт сами *признаки* (те же данные обучения, те же отсечки), поэтому он должен переноситься лучше — но при +0.0005 эффект всё ещё может быть в пределах шума публичного теста.